Tópicos Avançados em IA

Rayssa Coelho(BEC): Dataset Dermatology

1. Setup e Importações

In [7]:
!pip install -r requirements_av.txt

In [17]:
import pandas as pd          #manipular dados (dataframe)
import numpy as np           #cálculos
import wandb                 #rastrear, visualizar e gerenciar experimentos de ML
import kagglehub             #acessar datasets do Kaggle
import shutil # serve para mexer com arquivos e pastas
import os     # para lidar com caminhos de arquivos e diretórios
from dotenv import load_dotenv # para carregar variáveis de ambiente de um arquivo .env

In [ ]:
import matplotlib.pyplot as plt #gráficos
import seaborn as sns #gráficos
import random # para gerar números aleatórios
from sklearn.model_selection import train_test_split # para dividir os dados em treino e teste
from sklearn.impute import SimpleImputer # para tratar valores faltantes
from sklearn.feature_selection import mutual_info_regression # para calcular a importância das features usando mutual information
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

2. Adicionando o Dataset

In [9]:
import kagglehub # Download da ultima versão do kagglehub 
path = kagglehub.dataset_download("olcaybolat1/dermatology-dataset-classification") 
print("Path to dataset files:", path)

Path to dataset files: C:\Users\rayco\.cache\kagglehub\datasets\olcaybolat1\dermatology-dataset-classification\versions\5


In [11]:
import shutil # serve para mexer com arquivos e pastas
import os     # para lidar com caminhos de arquivos e diretórios
from dotenv import load_dotenv # para carregar variáveis de ambiente de um arquivo .env

# carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# caminho que o kagglehub retornou
source_path = path  

# destino dentro do seu projeto
dest_path = os.path.join(os.getcwd(), "data")

# copia tudo pra pasta data
shutil.copytree(source_path, dest_path, dirs_exist_ok=True)

print("Dataset copiado para:", dest_path)

Dataset copiado para: c:\Users\rayco\Desktop\atividades\aulas_ia\learning_ai\projeto_avaliacao\data


In [12]:
df_raw = pd.read_csv( # lê o arquivo CSV
     "data/dermatology_database_1.csv",
    low_memory=False # evita avisos de tipos de dados mistos
)
print(f"Shape: {df_raw.shape}") # mostra o número de linhas e colunas
print(f"Columns: {list(df_raw.columns[:5])}...") # mostra os nomes das primeiras 5 colunas
df_raw.head() # mostra as primeiras 5 linhas do dataframe

Shape: (366, 35)
Columns: ['erythema', 'scaling', 'definite_borders', 'itching', 'koebner_phenomenon']...


,erythema,scaling,definite_borders,itching,koebner_phenomenon,polygonal_papules,follicular_papules,oral_mucosal_involvement,knee_and_elbow_involvement,scalp_involvement,...,disappearance_granular_layer,vacuolisation_damage_basal_layer,spongiosis,saw_tooth_appearance_retes,follicular_horn_plug,perifollicular_parakeratosis,inflammatory_mononuclear_infiltrate,band_like_infiltrate,age,class
0,2,2,0,3,0,0,0,0,1,0,...,0,0,3,0,0,0,1,0,55,2
1,3,3,3,2,1,0,0,0,1,1,...,0,0,0,0,0,0,1,0,8,1
2,2,1,2,3,1,3,0,3,0,0,...,0,2,3,2,0,0,2,3,26,3
3,2,2,2,0,0,0,0,0,3,2,...,3,0,0,0,0,0,3,0,40,1
4,2,3,2,2,2,2,0,2,0,0,...,2,3,2,3,0,0,2,3,45,3


3. Login no W&B e Criação Artefato

In [14]:
import os
from dotenv import load_dotenv # para carregar variáveis de ambiente de um arquivo .env
import wandb

API_KEY = os.getenv("WANDB_API_KEY") # pega a chave da API do W&B do arquivo .env
wandb.login(key=API_KEY) 

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [15]:
# Inicializa um novo run no W&B para esta etapa de carregamento dos dados
wandb.init(
    project="mlops-project-dermatology", # nome do projeto no W&B
    job_type="load_raw", # tipo do job (pode ser "load_raw", "preprocess", "train", etc.)
    name="load_raw" # nome do run (pode ser "load_raw", "preprocess", "train", etc.
)

# Cria um novo artifact para o dataset bruto
artifact = wandb.Artifact( # cria um novo artifact
    name="raw_data", # artifact name (versioned automatically
    type="dataset", # tipo do artifact (pode ser "dataset", "model", "code", etc.)
    description="Dermatology raw dataset from Kaggle"
)
temp_path = "temp_raw.csv" # caminho temporário para salvar o CSV do dataframe
df_raw.to_csv(temp_path, index=False) # salva o dataframe como CSV sem o índice
artifact.add_file(temp_path) # adiciona o arquivo CSV ao artifact

# Log the artifact to W&B
wandb.log_artifact(artifact) # faz o upload do artifact para o W&B

# Log summary statistics visible on the W&B dashboard
wandb.summary["rows"] = len(df_raw) # número de linhas
wandb.summary["columns"] = list(df_raw.columns) # lista de colunas (pode ser truncada se for muito longa)
wandb.finish() # finaliza o run atual no W&B

print("Raw data artifact saved to W&B.")

rows,366


Raw data artifact saved to W&B.


Obs: O arquivo temporário (temp_raw.csv) é criado porque o método wandb.Artifact.add_file() espera um caminho para um arquivo no sistema de arquivos local. O df_raw é um objeto pandas.DataFrame que existe apenas na memória. Para que o Weights & Biases possa rastrear e versionar esse DataFrame como um artefato, ele precisa primeiro ser salvo em um arquivo. Criar um arquivo temporário é uma prática comum para fazer isso sem sobrecarregar o diretório de trabalho com arquivos intermediários permanentes.

4. Limpeza de Dados

In [ ]:
# Remoção de duplicatas e tratamento de valores faltantes

def remove_duplicates(df): # remove linhas duplicadas do dataframe
    before = len(df)        # salva número original de linhas
    df = df.drop_duplicates() # remove todas as linhas duplicadas do dataframe
    print(f"Removed {before - len(df)} duplicates") # imprime quantas linhas foram removidas
    return df # retorna o dataframe sem duplicatas

def handle_missing_values(df, strategy='mean', threshold=0.5): # trata valores faltantes
    missing_frac = df.isnull().mean() # calcula a fração de valores faltantes por coluna
    cols_to_drop = missing_frac[missing_frac > threshold].index.tolist() # identifica colunas com mais de 50% de valores faltantes
    df = df.drop(columns=cols_to_drop) # remove essas colunas do dataframe
    print(f"Dropped columns: {cols_to_drop}") # imprime quais colunas foram removidas 
    numeric_cols = df.select_dtypes(include=[np.number]).columns # seleciona apenas as colunas numéricas
    imputer = SimpleImputer(strategy=strategy) # cria um imputer para preencher os valores faltantes usando a estratégia escolhida (média, mediana, etc.)
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols]) # preenche os valores faltantes nas colunas numéricas usando a estratégia escolhida (média, mediana, etc.)
    cat_cols = df.select_dtypes(include=['object']).columns # seleciona apenas as colunas do tipo objeto
    df[cat_cols] = df[cat_cols].fillna('missing') # preenche os valores faltantes com 'missing'
    return df
#executando
df_clean = remove_duplicates(df_raw) # remove linhas duplicadas
df_clean = handle_missing_values(df_clean, threshold=0.5) # trata valores faltantes
print(f"Cleaned dataset shape: {df_clean.shape}") # mostra o número de linhas e colunas do dataset limpo

Removed 0 duplicates
Dropped columns: []
Cleaned dataset shape: (366, 35)


C:\Users\rayco\AppData\Local\Temp\ipykernel_5020\2169975705.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns # seleciona apenas as colunas do tipo objeto
